In [29]:
import pandas as pd
import logging

# Aqui se configura el logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

def limpiar_datos(df):
    """Aqui se limpia y convierte los datos en los tipos correctos."""
    
    # Aqui se maneja los valores con "_x000D_" en strings
    df = df.applymap(lambda x: x.replace('_x000D_', '') if isinstance(x, str) else x)

     # Aqui se convierte fecha y hora correctamente
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["hora"] = pd.to_datetime(df["hora"], format="%H:%M:%S", errors="coerce").dt.time
    df["hora"] = df["hora"].fillna(pd.to_datetime("00:00:00").time())  # Aqui se asigna "00:00:00" por defecto

    # Aqui se convierte `ultmodificacion`
    df["ultmodificacion"] = pd.to_datetime(df["ultmodificacion"], errors="coerce")

    # Aquie se convierte a tipos numéricos seguros con valores por defecto
    columnas_enteras = ["punto", "idcadena", "id"]
    for col in columnas_enteras:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("Int32")
    
    # Aqui se convierte ticket a string para preservar el formato con guiones
    df["ticket"] = df["ticket"].astype(str)

    # Aqui se convierte unidades_vendidas a float (manejar negativos)
    df["unidades_vendidas"] = pd.to_numeric(df["unidades_vendidas"], errors="coerce").fillna(0).astype(float)
    
    # Aqui se convierte precio_regular y precio_promocional a float con 0.0 por defecto
    df["precio_regular"] = pd.to_numeric(df["precio_regular"], errors="coerce").fillna(0.0).astype(float)
    df["precio_promocional"] = pd.to_numeric(df["precio_promocional"], errors="coerce").fillna(0.0).astype(float)

    # Aqui se convierte Convertir anulado a booleano
    df["anulado"] = df["anulado"].astype(str).str.lower().map({"true": True, "false": False}).fillna(False)

    # Aqui se asegura los valores por defecto en columnas de texto
    columnas_texto = ["eancode", "ean_desc", "tipo_venta"]
    for col in columnas_texto:
        df[col] = df[col].fillna("")

    return df

def alinear_tickets_pandas(ruta_archivo_entrada, ruta_archivo_salida, ruta_archivo_incorrectas):

    """Aqui se carga, limpia y guarda el archivo procesado."""
    try:
        logging.info("Cargando archivo...")
        df = pd.read_csv(ruta_archivo_entrada, delimiter=";", encoding="utf-8", low_memory=False)
        
        # Aplicar limpieza
        df = limpiar_datos(df)

        # Separar datos correctos e incorrectos
        df_correctos = df.dropna()
        df_incorrectos = df[df.isna().any(axis=1)]
        
        # Guardar archivos
        df_correctos.to_csv(ruta_archivo_salida, sep=';', index=False, encoding='utf-8')
        df_incorrectos.to_csv(ruta_archivo_incorrectas, sep=';', index=False, encoding='utf-8')
        
        logging.info(f"✅ Archivo limpio guardado en: {ruta_archivo_tickets_salida} ({len(df_correctos)} registros)")
        logging.info(f"❌ Registros incorrectos guardados en: {ruta_archivo_tickets_incorrectas} ({len(df_incorrectos)} registros)")

    except Exception as e:
        logging.error(f"⚠️ Error inesperado: {e}")

# Aqui se asignan las rutas de los archivos 
ruta_archivo_tickets = 'C:/Users/MATI_/OneDrive/Escritorio/Assessment_Data_Engineer/Descargas_Drive/tickets_prueba_v2.txt'
ruta_archivo_tickets_salida = 'C:/Users/MATI_/OneDrive/Escritorio/Assessment_Data_Engineer/Archivos_verificados/tickets_correctos.txt'
ruta_archivo_tickets_incorrectas = 'C:/Users/MATI_/OneDrive/Escritorio/Assessment_Data_Engineer/Archivos_verificados/tickets_incorrectos.txt'


# Aqui se ejecuta el procesamiento
alinear_tickets_pandas(ruta_archivo_tickets, ruta_archivo_tickets_salida, ruta_archivo_tickets_incorrectas)



📂 Cargando archivo...
✅ Archivo limpio guardado en: C:/Users/MATI_/OneDrive/Escritorio/Assessment_Data_Engineer/Archivos_verificados/tickets_alineado.txt (4695207 registros)
❌ Registros incorrectos guardados en: C:/Users/MATI_/OneDrive/Escritorio/Assessment_Data_Engineer/Archivos_verificados/tickets_incorrecto.txt (0 registros)
